# HA/DR & Replication Tests

## Test Cases (Public Preview)
| Capability | Test Case | Target |
|------------|-----------|--------|
| Cross-Region Replication | Create replication group, insert 1M rows | RPO <1hr under 100 TPS |
| Failover Test | Simulate outage, execute failover | RTO <4hr |
| Metadata Validation | Query SYSTEM$GET_ICEBERG_TABLE_INFORMATION | Verify paths, test R/W |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

## Test 1: Cross-Region Replication (Public Preview)
Create replication group and measure lag under load.

In [ ]:
-- Note: Cross-region replication requires secondary account setup
-- This is a template for the replication configuration

-- Step 1: Create replication group (run on PRIMARY account)
-- CREATE REPLICATION GROUP ICEBERG_POC_RG
--     OBJECT_TYPES = DATABASES
--     ALLOWED_DATABASES = ICEBERG_POC
--     ALLOWED_ACCOUNTS = <SECONDARY_ACCOUNT_LOCATOR>
--     REPLICATION_SCHEDULE = '10 MINUTE';

-- Step 2: Create secondary replication group (run on SECONDARY account)
-- CREATE REPLICATION GROUP ICEBERG_POC_RG
--     AS REPLICA OF <PRIMARY_ORG>.<PRIMARY_ACCOUNT>.ICEBERG_POC_RG;

-- Check replication status
SHOW REPLICATION GROUPS;

In [ ]:
-- Create Iceberg table for replication testing
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.REPLICATION_TEST (
    id INT,
    data STRING,
    created_at TIMESTAMP_NTZ(9)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

-- Insert 1M rows for replication load test
INSERT INTO ICEBERG_POC.TESTS.REPLICATION_TEST
SELECT 
    SEQ4() AS id,
    'Data_' || SEQ4() AS data,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ(9) AS created_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000000));

SELECT COUNT(*) AS row_count FROM ICEBERG_POC.TESTS.REPLICATION_TEST;

## Test 2: Metadata Validation
Query SYSTEM$GET_ICEBERG_TABLE_INFORMATION and verify metadata paths.

In [ ]:
-- Get Iceberg table information including metadata paths
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.REPLICATION_TEST') AS iceberg_info;

-- Parse and display metadata details
SELECT 
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.REPLICATION_TEST')):metadataLocation::STRING AS metadata_location,
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.REPLICATION_TEST')):status::STRING AS status;

-- Show all Iceberg tables and their metadata
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_POC.TESTS;

## Test 3: Failover Simulation
Document failover procedure and measure RTO/RPO targets (RTO < 4hr, RPO < 1hr).

In [ ]:
-- Failover Procedure Template (run on SECONDARY account during DR event)
-- 
-- Step 1: Check replication lag before failover
-- SELECT REPLICATION_GROUP_NAME, LAST_REFRESH_TIME, NEXT_SCHEDULED_REFRESH_TIME
-- FROM TABLE(INFORMATION_SCHEMA.REPLICATION_GROUP_REFRESH_HISTORY())
-- ORDER BY LAST_REFRESH_TIME DESC LIMIT 5;
--
-- Step 2: Execute failover (promotes secondary to primary)
-- ALTER REPLICATION GROUP ICEBERG_POC_RG FAILOVER;
--
-- Step 3: Verify Iceberg table accessibility
-- SELECT COUNT(*) FROM ICEBERG_POC.TESTS.REPLICATION_TEST;
--
-- Step 4: Test R/W operations
-- INSERT INTO ICEBERG_POC.TESTS.REPLICATION_TEST VALUES (999999, 'Failover_Test', CURRENT_TIMESTAMP());
-- SELECT * FROM ICEBERG_POC.TESTS.REPLICATION_TEST WHERE id = 999999;

-- Record failover metrics
-- RTO Target: < 4 hours
-- RPO Target: < 1 hour

SELECT 'Failover procedure documented - requires secondary account for testing' AS status;